In [54]:
import torch
import matplotlib.pyplot as plt
import pandas as pd

In [91]:
mens = open('data/men.txt').read().splitlines()
womens = open('data/women.txt').read().splitlines()

df = pd.DataFrame(mens + womens, columns=['name'])

df['name'] = df['name'].str.lower()
df = df[~df['name'].str.endswith('.')]
df = df[df['name'].str.len() >= 3]
df = df.drop_duplicates()
df = df.sample(frac=1.0, random_state=42)

In [92]:
df.describe()

,name
count,23184
unique,23184
top,burcu
freq,1


In [93]:
names = df['name'].to_list()
names[:10]

['burcu',
 'tuure',
 'reemus',
 'charissa',
 'leandro',
 'fuaad',
 'mahi',
 'danni',
 'suaad',
 'davinder']

In [94]:
vocabulary = sorted(list(set(''.join(names))))
vocabulary.insert(0, '<S>')
vocabulary.insert(1, '<E>')
len(vocabulary)

54

In [95]:
atoi = {c: i for i, c in enumerate(vocabulary)}
itoa = {i: c for i, c in enumerate(vocabulary)}
atoi

{'<S>': 0,
 '<E>': 1,
 "'": 2,
 '-': 3,
 'a': 4,
 'b': 5,
 'c': 6,
 'd': 7,
 'e': 8,
 'f': 9,
 'g': 10,
 'h': 11,
 'i': 12,
 'j': 13,
 'k': 14,
 'l': 15,
 'm': 16,
 'n': 17,
 'o': 18,
 'p': 19,
 'q': 20,
 'r': 21,
 's': 22,
 't': 23,
 'u': 24,
 'v': 25,
 'w': 26,
 'x': 27,
 'y': 28,
 'z': 29,
 'à': 30,
 'á': 31,
 'â': 32,
 'ã': 33,
 'ä': 34,
 'å': 35,
 'ç': 36,
 'è': 37,
 'é': 38,
 'ê': 39,
 'ë': 40,
 'í': 41,
 'î': 42,
 'ï': 43,
 'ò': 44,
 'ó': 45,
 'ô': 46,
 'õ': 47,
 'ö': 48,
 'ø': 49,
 'ù': 50,
 'ú': 51,
 'ü': 52,
 'þ': 53}

In [96]:
block_size = 8 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w:
      ix = atoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append
    context = context[1:] + [1] # add <E> at the end of the word

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

n1 = int(0.8*len(names))
n2 = int(0.9*len(names))
Xtr,  Ytr  = build_dataset(names[:n1])     # 80%
Xdev, Ydev = build_dataset(names[n1:n2])   # 10%
Xte,  Yte  = build_dataset(names[n2:])     # 10%

torch.Size([127803, 8]) torch.Size([127803])
torch.Size([16156, 8]) torch.Size([16156])
torch.Size([16114, 8]) torch.Size([16114])


In [98]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itoa[int(ix.item())] for ix in x), '-->', itoa[int(y.item())])

<S><S><S><S><S><S><S><S> --> b
<S><S><S><S><S><S><S>b --> u
<S><S><S><S><S><S>bu --> r
<S><S><S><S><S>bur --> c
<S><S><S><S>burc --> u
<S><S><S><S><S><S><S><S> --> t
<S><S><S><S><S><S><S>t --> u
<S><S><S><S><S><S>tu --> u
<S><S><S><S><S>tuu --> r
<S><S><S><S>tuur --> e
<S><S><S><S><S><S><S><S> --> r
<S><S><S><S><S><S><S>r --> e
<S><S><S><S><S><S>re --> e
<S><S><S><S><S>ree --> m
<S><S><S><S>reem --> u
<S><S><S>reemu --> s
<S><S><S><S><S><S><S><S> --> c
<S><S><S><S><S><S><S>c --> h
<S><S><S><S><S><S>ch --> a
<S><S><S><S><S>cha --> r
